# Phase 15: A4 Explainability, Grad-CAM, and Morphological Interpretability

All non-training operations are explicitly authorized. Gradients are used only for explanation; no optimizer is created and model parameters are frozen.

Primary layer stage3 (128x16x16) is the late encoder before both global/local paths and offers more spatial detail than stage4 (192x8x8), which is secondary. Selection is architectural, before mask-overlap analysis. Use predicted-class logits for every slice and additionally true-class logits for errors. Apply spatial mean gradients, weighted feature sum and ReLU, bilinear upsampling with align_corners=False directly to original image size, then divide by positive peak (no minimum subtraction). Zero-positive maps receive zero localization scores and are explicitly counted, not excluded. Peak ties use the first maximum in row-major order. Ratio epsilon is 1e-8. Fixed heatmap thresholds 0.3/0.5/0.7 are descriptive, never selected for best overlap. Original-image masks evaluate Grad-CAM; auxiliary morphology is evaluated on frozen 256x256 targets. Region/boundary logits use sigmoid and threshold 0.5; distance is already sigmoid in the model. Boundary F1 uses Euclidean tolerance 2 pixels on the 256x256 grid, no physical-millimeter interpretation. Primary results are means of per-slice metrics; patient summaries average slices within each patient. Cluster bootstrap resamples patients, seed 42, 10000 iterations. No significance tests.

Representative selection is fixed before Grad-CAM: for each true class, highest-confidence correct, lowest-confidence correct, seeded random correct, highest-confidence error, and seeded random error; numeric Sample ID breaks confidence ties, and previously used patients are avoided when possible. These selections never inspect heatmaps or mask overlap. Additional failure illustrations deliberately show the minimum and maximum stage3 attention mass among errors of each class, with explicit extreme-case labels; these are diagnostic extremes, never representative performance evidence. No PID appears in figure labels. A5 comparison is omitted to keep the analysis focused on A4 and avoid doubling checkpoint/explanation execution for an optional, non-primary question.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.


In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
import sys
ROOT=Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.explainability.pipeline import run, SETTINGS
from src.explainability.core import synthetic_tests
CONFIRM_EXPLAINABILITY_ANALYSIS=False
assert CONFIRM_EXPLAINABILITY_ANALYSIS
print(SETTINGS)
synthetic_tests()

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
OUTPUT=run(ROOT)
print(OUTPUT)

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from IPython.display import display, Markdown, Image
from src.phase14_statistics import read
import pandas as pd
pd.set_option('display.max_rows',300)
pd.set_option('display.max_columns',20)
for name in ['checkpoint_audit','prediction_reproducibility','gradcam_localization_summary','gradcam_per_class_summary','gradcam_correct_vs_incorrect','morphology_summary','patient_morphology_attention_correlations','failure_pattern_counts','zero_map_audit','explainability_fold_summary']:
    display(Markdown('### '+name.replace('_',' ').title()))
    display(read(OUTPUT/'tables'/f'{name}.csv'))

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
for name in ['figure1_representative_gradcam','figure2_correct_vs_incorrect','figure3_morphology_predictions','figure4_quantitative_localization','figure5_per_class_explainability']:
    display(Image(filename=str(OUTPUT/'figures'/f'{name}.png')))
display(Markdown((OUTPUT/'PUBLICATION_EXPLAINABILITY_SUBSECTION.md').read_text()))

## Scientific limitations

Grad-CAM is a post-hoc, coarse, class-conditioned localization method, not a segmentation algorithm or proof of causal feature importance. Overlap with tumor annotations does not establish clinical validity, and a confident, incorrect classifier may attend inside the tumor. Spatial resolution is limited to 16x16 (primary) or 8x8 (secondary); interpolation does not add independent detail. MRI background, normalization and display contrast can affect interpretation. Masks are available only as evaluation targets, never model inference inputs. Slices within a patient are correlated; patient-cluster intervals are conditional on fixed trained models, omit training randomness, and do not fully capture overlapping CV training-set dependence. No independent external or prospective clinical dataset is available. A4 was selected after Phase 13 results, so these findings are exploratory and cannot tune or validate that selection. Region/boundary/distance heads are auxiliary predictions, not a separately validated clinical segmentation system. Mean distance errors over the full grid may be dominated by background, so tumor-only MAE is also reported. Fixed CAM thresholds are sensitivity settings, not test-optimized operating points. Ratios can be large when outside activation is tiny; epsilon and zero-map counts are disclosed. Primary correct/incorrect grouping uses slice correctness; patient-equal summaries average within the relevant slice subgroup, not patient-level classification correctness. Extreme failure examples are explicitly selected extremes, not representative prevalence estimates. No A5 explanation comparison was executed; it was optional and omitted to keep the scope focused and avoid additional checkpoint/compute complexity. Phase 16 was not started.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
import json
audit=json.loads((OUTPUT/'final_scientific_audit.json').read_text())
assert audit['status']=='PASS' and audit['slices']==3064 and audit['patients']==233
assert audit['optimizer_steps']==0 and audit['weights_and_buffers_unchanged']
assert audit['protected_files_unchanged'] and not audit['phase16_started']
display(audit)
print('Phase 15 completed. STOP before Phase 16.')